# Derivative of the matter power spectrum

This notebook computes $\partial P(k) / \partial \Omega_m$ today by forward-mode automatic differentiation and checks it against a finite difference.

In [1]:
import jax
import jax.numpy as jnp
import numpy as np

from discoeb import LCDM, matter_power_spectrum, solve_eb

K = np.geomspace(1.0e-4, 1.0, 64)  # wave numbers, 1/Mpc

Let's set up a flat Planck 2018 best fit cosmology with three massless neutrino flavours, given by $\Omega_m$, $\Omega_b$ and $H_0$:

In [2]:
h = 0.6732117
PLANCK = dict(
    Omega_m=(0.02238280 + 0.1201075) / h**2,
    Omega_b=0.02238280 / h**2,
    H_0=100.0 * h,
    A_s=2.100549e-9,
    n_s=0.9660499,
    T_cmb=2.7255,
    Y_He=0.2454006,
    N_eff=3.046,
)

`solve_matter_power_spectrum` maps $\Omega_m$ to $P(k)$ today. We hold $\Omega_b$, $H_0$ and flatness fixed, so more matter means less dark energy.

In [3]:
def solve_matter_power_spectrum(Omega_m):
    """``P(k)`` today in Mpc^3, ``(len(K),)``, of the Planck cosmology at this Omega_m."""

    cosmology = LCDM.from_density_parameters(**{**PLANCK, "Omega_m": Omega_m})
    values = solve_eb(
        cosmology, K, [1.0], keep=("delta_c", "delta_b")
    )  # (cosmology, name, k, a)
    delta_c, delta_b = jnp.unstack(values[0, :, :, -1])
    return matter_power_spectrum(
        K,
        delta_c=delta_c,
        delta_b=delta_b,
        omega_c=cosmology.omega_c,
        omega_b=cosmology.omega_b,
        A_s=cosmology.A_s,
        n_s=cosmology.n_s,
        k_p=cosmology.k_p,
    )

`jax.jacfwd` of `solve_matter_power_spectrum` gives $\partial P / \partial \Omega_m$ at every wave number from one forward-sensitivity integration. The derivatives are solved at the fixed scale factors supplied to `solve_eb`, in this case just $a = 1$, today.

In [4]:
Omega_m = jnp.asarray(PLANCK["Omega_m"])
pk = solve_matter_power_spectrum(Omega_m)
dpk = jax.jacfwd(solve_matter_power_spectrum)(Omega_m)
slope = np.asarray(Omega_m * dpk / pk)  # d ln P / d ln Omega_m

Let's check the result against finite differences:

In [5]:
step = 1.0e-3
finite = np.asarray(
    Omega_m
    * jnp.log(
        solve_matter_power_spectrum(Omega_m + step)
        / solve_matter_power_spectrum(Omega_m - step)
    )
    / (2.0 * step)
)

The two slopes agree to better than $10^{-2}$.

In [6]:
print(
    f"{'k [1/Mpc]':>12}  {'P(k) [Mpc^3]':>14}  {'dlnP/dlnOm':>11}  {'finite diff.':>12}"
)
for i in range(0, len(K), 7):
    print(f"{K[i]:12.4e}  {pk[i]:14.6e}  {slope[i]:11.5f}  {finite[i]:12.5f}")
print(f"largest difference between the two slopes: {np.abs(slope - finite).max():.1e}")

   k [1/Mpc]    P(k) [Mpc^3]   dlnP/dlnOm  finite diff.
  1.0000e-04    2.017416e+03     -1.53710      -1.53739
  2.7826e-04    5.402943e+03     -1.53109      -1.53112
  7.7426e-04    1.421704e+04     -1.49816      -1.49821
  2.1544e-03    3.486639e+04     -1.37569      -1.37579
  5.9948e-03    6.883193e+04     -1.03789      -1.03740
  1.6681e-02    7.392893e+04     -0.16225      -0.16254
  4.6416e-02    3.258168e+04      0.74347       0.74308
  1.2915e-01    7.019345e+03      1.50650       1.50643
  3.5938e-01    9.229124e+02      2.13956       2.13967
  1.0000e+00    9.044880e+01      2.42820       2.42837
largest difference between the two slopes: 3.0e-03
